In [17]:
# =============================================================================
# HARYANA WATER QUALITY — HEALTH ALERT & RISK MANAGEMENT SYSTEM
# =============================================================================
# Components:
#   1. Health data entry (based on 30-question questionnaire)
#   2. Risk scoring engine (3 triggers: WQI site, symptoms, sanitation)
#   3. Email alerts → affected person + doctor/health authority
#   4. PDF health risk report per flagged individual
#   5. Dashboard — visual overview of all high-risk individuals
#
# Setup (run once):
#   pip install reportlab matplotlib pandas
#
# Email setup:
#   Set these environment variables (never hardcode credentials):
#     ALERT_EMAIL_SENDER   = your Gmail address
#     ALERT_EMAIL_PASSWORD = your Gmail App Password (not login password)
#                            → Gmail → Manage Account → Security → App Passwords
#     AUTHORITY_EMAIL      = doctor/health authority email
#
# =============================================================================

In [18]:
import os
import smtplib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                 TableStyle, HRFlowable, PageBreak)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

# 
# WQI SITE LOOKUP (from PROJECT REPORT data)
# Maps site name keywords - WQI class
# Updated automatically if you pass the actual df from your notebooks

SITE_WQI_CLASS = {
    # These will be overridden by actual computed values at runtime
    # Default placeholders based on known pollution levels in literature
    "Kaushalaya Dam":    "Good",
    "Nada Sahib":        "Good",
    "Rajpura Road":      "Poor",
    "Ratia":             "Very Poor",
    "Ottu Barrage":      "Very Poor",
    "Yamunanagar":       "Poor",
    "Panipat":           "Poor",
    "Sonipat":           "Very Poor",
    "Faridabad":         "Unsuitable",
    "Shahabad":          "Poor",
    "Agra Canal":        "Very Poor",
    "Teekar Taal":       "Good",
    "Karan Lake":        "Good",
    "Brahma Sarovar":    "Poor",
}

HIGH_RISK_WQI_CLASSES = {"Poor", "Very Poor", "Unsuitable"}


# RISK SCORING CONFIG

SYMPTOM_RISK_WEIGHTS = {
    # High severity symptoms (water-borne / pollution linked)
    "diarrhea":              3,
    "diarrhea_blood":        5,
    "kidney_disease":        5,
    "jaundice":              5,
    "dysentery":             5,
    "fever":                 2,
    "anemia":                3,
    "high_blood_pressure":   3,
    "vomiting":              2,
    "abdominal_pain":        2,
    "nausea":                2,
    "weight_loss":           2,
    "painful_skin_lesions":  4,   # fluoride/arsenic related
    "bone_disease":          4,   # fluoride related
    "malnutrition":          3,
    "growth_retardation":    3,
    "gastric_problems":      2,
    "severe_throat":         2,
    "depression":            1,
    "menstrual_problems":    2,
}

HIGH_RISK_SCORE_THRESHOLD = 6   # total symptom score to trigger alert
SYMPTOM_COUNT_THRESHOLD   = 2   # min number of distinct symptoms


def compute_risk(person: dict) -> dict:
    """
    Compute risk score for one person.
    Returns dict with: risk_score, risk_level, triggers, symptom_score
    """
    triggers      = []
    symptom_score = 0
    symptom_count = 0

    #  Trigger 1: Lives near bad WQI site 
    site = str(person.get("nearest_site", "")).strip()
    wqi_class = "Unknown"
    for key, cls in SITE_WQI_CLASS.items():
        if key.lower() in site.lower():
            wqi_class = cls
            break
    person["_wqi_class"] = wqi_class
    if wqi_class in HIGH_RISK_WQI_CLASSES:
        triggers.append(f"Lives near {wqi_class} WQI site ({site})")

    #  Trigger 2: No access to clean water / sanitation 
    water_source = str(person.get("drinking_water_source", "")).lower()
    water_quality = str(person.get("drinking_water_quality", "")).lower()
    has_toilet    = str(person.get("has_toilet", "yes")).lower()
    uses_purifier = str(person.get("uses_purifier", "yes")).lower()

    if "river" in water_source or "well" in water_source:
        triggers.append("Drinking from unprotected water source (river/open well)")
    if "unsatisfactory" in water_quality or "unfiltered" in water_quality:
        triggers.append("Drinking water quality reported as unsatisfactory/unfiltered")
    if has_toilet in ("no", "false", "0"):
        triggers.append("No access to private toilet (open defecation)")
    if uses_purifier in ("no", "false", "0") and wqi_class in HIGH_RISK_WQI_CLASSES:
        triggers.append("No water purifier used despite living near polluted site")

    # Trigger 3: Reported symptoms 
    reported_symptoms = []
    for symptom, weight in SYMPTOM_RISK_WEIGHTS.items():
        val = person.get(symptom, False)
        if val in (True, "yes", "Yes", 1, "1", "true", "True"):
            symptom_score += weight
            symptom_count += 1
            reported_symptoms.append(symptom.replace("_", " ").title())

    person["_reported_symptoms"] = reported_symptoms
    person["_symptom_score"]     = symptom_score

    if symptom_count >= SYMPTOM_COUNT_THRESHOLD:
        triggers.append(
            f"{symptom_count} health symptoms reported "
            f"(severity score: {symptom_score})"
        )

    #  Overall risk level 
    n_triggers = len(triggers)
    if n_triggers >= 3 or symptom_score >= HIGH_RISK_SCORE_THRESHOLD:
        risk_level = "CRITICAL"
    elif n_triggers == 2 or symptom_score >= 4:
        risk_level = "HIGH"
    elif n_triggers == 1:
        risk_level = "MODERATE"
    else:
        risk_level = "LOW"

    return {
        "risk_level":   risk_level,
        "risk_score":   n_triggers * 10 + symptom_score,
        "triggers":     triggers,
        "symptom_score": symptom_score,
        "wqi_class":    wqi_class,
    }

In [19]:
# PDF REPORT GENERATOR

RISK_COLORS = {
    "CRITICAL": colors.HexColor("#c0392b"),
    "HIGH":     colors.HexColor("#e67e22"),
    "MODERATE": colors.HexColor("#f1c40f"),
    "LOW":      colors.HexColor("#27ae60"),
}

def generate_pdf_report(person: dict, risk: dict, output_path: str):
    """Generate a PDF health risk report for one individual."""
    doc    = SimpleDocTemplate(output_path, pagesize=A4,
                               topMargin=1.5*cm, bottomMargin=1.5*cm,
                               leftMargin=2*cm, rightMargin=2*cm)
    styles = getSampleStyleSheet()
    story  = []

    #  Custom styles 
    title_style = ParagraphStyle("title", parent=styles["Title"],
                                  fontSize=16, spaceAfter=4,
                                  textColor=colors.HexColor("#1a252f"),
                                  alignment=TA_CENTER)
    subtitle_style = ParagraphStyle("subtitle", parent=styles["Normal"],
                                     fontSize=10, spaceAfter=2,
                                     textColor=colors.HexColor("#555"),
                                     alignment=TA_CENTER)
    section_style = ParagraphStyle("section", parent=styles["Heading2"],
                                    fontSize=11, spaceBefore=12, spaceAfter=4,
                                    textColor=colors.HexColor("#2c3e50"),
                                    borderPad=2)
    body_style = ParagraphStyle("body", parent=styles["Normal"],
                                 fontSize=9.5, leading=14,
                                 textColor=colors.HexColor("#333"))
    risk_color = RISK_COLORS.get(risk["risk_level"], colors.grey)

    # Header 
    story.append(Paragraph("HARYANA WATER QUALITY HEALTH RISK REPORT", title_style))
    story.append(Paragraph("Groundwater & Surface Water Monitoring Programme", subtitle_style))
    story.append(Paragraph(f"Generated: {datetime.now().strftime('%d %B %Y, %I:%M %p')}",
                            subtitle_style))
    story.append(HRFlowable(width="100%", thickness=2,
                             color=colors.HexColor("#2c3e50"), spaceAfter=10))

    # Risk level banner 
    banner_data = [[
        Paragraph(
            f'<font color="white"><b>RISK LEVEL: {risk["risk_level"]}'
            f'  |  Risk Score: {risk["risk_score"]}'
            f'  |  WQI Site Class: {risk["wqi_class"]}</b></font>',
            ParagraphStyle("banner", parent=styles["Normal"],
                            fontSize=11, alignment=TA_CENTER,
                            textColor=colors.white)
        )
    ]]
    banner_table = Table(banner_data, colWidths=["100%"])
    banner_table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), risk_color),
        ("PADDING",    (0,0), (-1,-1), 10),
        ("ROUNDEDCORNERS", [4]),
    ]))
    story.append(banner_table)
    story.append(Spacer(1, 12))

    #  Personal information 
    story.append(Paragraph("Personal Information", section_style))
    personal_data = [
        ["Name",          person.get("name", "—"),
         "Age",           str(person.get("age", "—"))],
        ["Gender",        person.get("gender", "—"),
         "Education",     person.get("education", "—")],
        ["Nearest Site",  person.get("nearest_site", "—"),
         "Family Size",   str(person.get("family_size", "—"))],
        ["Employment",    person.get("employment", "—"),
         "Email",         person.get("email", "—")],
    ]
    pt = Table(personal_data, colWidths=[3.5*cm, 5.5*cm, 3.5*cm, 5.5*cm])
    pt.setStyle(TableStyle([
        ("FONTNAME",    (0,0), (-1,-1), "Helvetica"),
        ("FONTSIZE",    (0,0), (-1,-1), 9),
        ("FONTNAME",    (0,0), (0,-1), "Helvetica-Bold"),
        ("FONTNAME",    (2,0), (2,-1), "Helvetica-Bold"),
        ("TEXTCOLOR",   (0,0), (0,-1), colors.HexColor("#2c3e50")),
        ("TEXTCOLOR",   (2,0), (2,-1), colors.HexColor("#2c3e50")),
        ("BACKGROUND",  (0,0), (0,-1), colors.HexColor("#ecf0f1")),
        ("BACKGROUND",  (2,0), (2,-1), colors.HexColor("#ecf0f1")),
        ("GRID",        (0,0), (-1,-1), 0.5, colors.HexColor("#bdc3c7")),
        ("PADDING",     (0,0), (-1,-1), 6),
        ("ROWBACKGROUNDS", (0,0), (-1,-1),
         [colors.white, colors.HexColor("#f8f9fa")]),
    ]))
    story.append(pt)
    story.append(Spacer(1, 8))

    # Sanitation & Water Access 
    story.append(Paragraph("Sanitation & Water Access", section_style))
    san_data = [
        ["Parameter", "Reported Value", "Status"],
        ["Drinking Water Source",
         person.get("drinking_water_source", "—"),
         "⚠ Risk" if any(x in str(person.get("drinking_water_source","")).lower()
                          for x in ["river","well"]) else "OK"],
        ["Water Quality",
         person.get("drinking_water_quality", "—"),
         "⚠ Risk" if "unsatisfactory" in str(person.get("drinking_water_quality","")).lower()
                  else "OK"],
        ["Has Private Toilet",
         str(person.get("has_toilet", "—")),
         "⚠ Risk" if str(person.get("has_toilet","yes")).lower() in ("no","false","0")
                  else "OK"],
        ["Uses Water Purifier",
         str(person.get("uses_purifier", "—")),
         "⚠ Risk" if str(person.get("uses_purifier","yes")).lower() in ("no","false","0")
                  else "OK"],
        ["Uses Disinfectant",
         str(person.get("uses_disinfectant", "—")), "—"],
        ["Washes Hands After Toilet",
         str(person.get("washes_hands", "—")), "—"],
    ]
    st = Table(san_data, colWidths=[6*cm, 6*cm, 6*cm])
    st.setStyle(TableStyle([
        ("FONTNAME",    (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE",    (0,0), (-1,-1), 9),
        ("BACKGROUND",  (0,0), (-1,0), colors.HexColor("#2c3e50")),
        ("TEXTCOLOR",   (0,0), (-1,0), colors.white),
        ("GRID",        (0,0), (-1,-1), 0.5, colors.HexColor("#bdc3c7")),
        ("PADDING",     (0,0), (-1,-1), 6),
        ("ROWBACKGROUNDS", (0,1), (-1,-1),
         [colors.white, colors.HexColor("#f8f9fa")]),
        ("TEXTCOLOR",   (2,1), (2,-1), colors.HexColor("#c0392b")),
    ]))
    story.append(st)
    story.append(Spacer(1, 8))

    #  Reported Symptoms 
    story.append(Paragraph("Reported Health Symptoms", section_style))
    symptoms = person.get("_reported_symptoms", [])
    if symptoms:
        sym_text = " | ".join(symptoms)
        story.append(Paragraph(
            f'<font color="#c0392b"><b>Symptoms ({len(symptoms)}):</b></font> {sym_text}',
            body_style))
    else:
        story.append(Paragraph("No symptoms reported.", body_style))

    story.append(Paragraph(
        f'<b>Symptom Severity Score:</b> {risk["symptom_score"]} / '
        f'{sum(SYMPTOM_RISK_WEIGHTS.values())}',
        body_style))
    story.append(Spacer(1, 8))

    #  Risk Triggers 
    story.append(Paragraph("Risk Triggers Identified", section_style))
    if risk["triggers"]:
        for i, t in enumerate(risk["triggers"], 1):
            story.append(Paragraph(f"{i}. {t}",
                ParagraphStyle("trigger", parent=body_style,
                                textColor=colors.HexColor("#c0392b"),
                                leftIndent=12, spaceAfter=3)))
    else:
        story.append(Paragraph("No risk triggers identified.", body_style))
    story.append(Spacer(1, 8))

    #  Recommendations 
    story.append(Paragraph("Recommendations", section_style))
    recs = _get_recommendations(risk, person)
    for rec in recs:
        story.append(Paragraph(
            f"• {rec}",
            ParagraphStyle("rec", parent=body_style,
                            leftIndent=12, spaceAfter=3)))
    story.append(Spacer(1, 8))

    # Footer 
    story.append(HRFlowable(width="100%", thickness=1,
                             color=colors.HexColor("#bdc3c7"), spaceBefore=8))
    story.append(Paragraph(
        "This report is auto-generated by the Haryana Water Quality Health "
        "Monitoring System. Please consult a qualified medical professional "
        "for diagnosis and treatment.",
        ParagraphStyle("footer", parent=styles["Normal"],
                        fontSize=8, textColor=colors.grey,
                        alignment=TA_CENTER)))

    doc.build(story)
    print(f" PDF saved: {output_path}")


def _get_recommendations(risk: dict, person: dict) -> list:
    recs = []
    if risk["wqi_class"] in HIGH_RISK_WQI_CLASSES:
        recs.append("Avoid using river or open well water for drinking and cooking.")
        recs.append("Install a certified water purifier (RO/UV) immediately.")
    if str(person.get("has_toilet","yes")).lower() in ("no","false","0"):
        recs.append("Access to sanitation facilities is critical — contact local Panchayat.")
    symptoms = person.get("_reported_symptoms", [])
    if "Kidney Disease" in symptoms or "Bone Disease" in symptoms:
        recs.append("High fluoride/nitrate levels linked to kidney and bone issues — "
                    "get water tested urgently.")
    if "Jaundice" in symptoms or "Diarrhea" in symptoms or "Dysentery" in symptoms:
        recs.append("Water-borne disease symptoms detected — seek medical attention immediately.")
    if "Anemia" in symptoms or "Weight Loss" in symptoms:
        recs.append("Nutritional deficiency suspected — consult a doctor for blood tests.")
    if risk["risk_level"] in ("CRITICAL", "HIGH"):
        recs.append("Visit the nearest Primary Health Centre (PHC) as soon as possible.")
        recs.append("Report water quality issues to Haryana State Pollution Control Board.")
    recs.append("Boil drinking water for at least 1 minute before consumption.")
    recs.append("Wash hands with soap before eating and after using toilet.")
    return recs

In [20]:
# EMAIL SENDER

def send_alert_email(person: dict, risk: dict, pdf_path: str):
    """Send alert emails to (1) the affected person and (2) health authority."""
    sender   = os.environ.get("ALERT_EMAIL_SENDER")
    password = os.environ.get("ALERT_EMAIL_PASSWORD")
    authority_email = os.environ.get("AUTHORITY_EMAIL", "health.authority@haryana.gov.in")

    if not sender or not password:
        print("  ⚠️  Email credentials not set. Skipping email.")
        print("     Set ALERT_EMAIL_SENDER and ALERT_EMAIL_PASSWORD env variables.")
        return

    risk_color_hex = {
        "CRITICAL": "#c0392b", "HIGH": "#e67e22",
        "MODERATE": "#f39c12", "LOW": "#27ae60"
    }.get(risk["risk_level"], "#555")

    # Email to affected person 
    person_email = person.get("email", "")
    if person_email:
        subject = (f"[{risk['risk_level']} RISK] Water Quality Health Alert — "
                   f"{person.get('name', 'Resident')}")

        triggers_html = "".join(
            f"<li>{t}</li>" for t in risk["triggers"]
        ) or "<li>No critical triggers</li>"
        symptoms_html = ", ".join(person.get("_reported_symptoms", [])) or "None reported"
        recs_html = "".join(
            f"<li>{r}</li>" for r in _get_recommendations(risk, person)
        )

        body_html = f"""
        <html><body style="font-family:Arial,sans-serif; color:#333; max-width:600px">
          <div style="background:{risk_color_hex}; padding:16px; border-radius:6px; color:white">
            <h2 style="margin:0">⚠ Health Risk Alert — {risk['risk_level']}</h2>
            <p style="margin:4px 0">Haryana Water Quality Health Monitoring System</p>
          </div>
          <div style="padding:16px">
            <p>Dear <b>{person.get('name','Resident')}</b>,</p>
            <p>Our water quality health monitoring system has identified you as a
            <b style="color:{risk_color_hex}">{risk['risk_level']} RISK</b> individual
            based on the water quality at your nearest sampling site
            (<b>{person.get('nearest_site','—')}</b>, WQI Class: <b>{risk['wqi_class']}</b>)
            and your reported health data.</p>

            <h3 style="color:#2c3e50">Risk Triggers:</h3>
            <ul>{triggers_html}</ul>

            <h3 style="color:#2c3e50">Reported Symptoms:</h3>
            <p>{symptoms_html}</p>

            <h3 style="color:#2c3e50">Recommendations:</h3>
            <ul>{recs_html}</ul>

            <p style="background:#fef9e7; padding:12px; border-left:4px solid #f39c12">
            Please find your detailed health risk report attached to this email.
            <b>Contact your nearest Primary Health Centre (PHC) immediately</b>
            if you are experiencing severe symptoms.</p>

            <p style="color:#888; font-size:12px">
            This is an automated alert. Please consult a qualified medical
            professional for diagnosis and treatment.</p>
          </div>
        </body></html>
        """
        _send_single_email(sender, password, person_email,
                           subject, body_html, pdf_path)
        print(f" Alert email sent to: {person_email}")

    # Email to health authority 
    authority_subject = (f"[HEALTH ALERT — {risk['risk_level']}] "
                         f"{person.get('name','Unknown')} | "
                         f"Site: {person.get('nearest_site','—')}")
    authority_html = f"""
    <html><body style="font-family:Arial,sans-serif; color:#333; max-width:600px">
      <div style="background:#2c3e50; padding:16px; border-radius:6px; color:white">
        <h2 style="margin:0">Health Authority Alert</h2>
        <p style="margin:4px 0">Haryana Water Quality Monitoring — Auto-generated</p>
      </div>
      <div style="padding:16px">
        <p>A <b style="color:{risk_color_hex}">{risk['risk_level']} RISK</b> individual
        has been identified:</p>
        <table style="border-collapse:collapse; width:100%">
          <tr><td style="padding:6px; background:#ecf0f1; font-weight:bold">Name</td>
              <td style="padding:6px">{person.get('name','—')}</td></tr>
          <tr><td style="padding:6px; background:#ecf0f1; font-weight:bold">Age / Gender</td>
              <td style="padding:6px">{person.get('age','—')} / {person.get('gender','—')}</td></tr>
          <tr><td style="padding:6px; background:#ecf0f1; font-weight:bold">Nearest Site</td>
              <td style="padding:6px">{person.get('nearest_site','—')} ({risk['wqi_class']})</td></tr>
          <tr><td style="padding:6px; background:#ecf0f1; font-weight:bold">Risk Score</td>
              <td style="padding:6px; color:{risk_color_hex}">
              <b>{risk['risk_score']} — {risk['risk_level']}</b></td></tr>
          <tr><td style="padding:6px; background:#ecf0f1; font-weight:bold">Symptoms</td>
              <td style="padding:6px">{', '.join(person.get('_reported_symptoms', [])) or 'None'}</td></tr>
          <tr><td style="padding:6px; background:#ecf0f1; font-weight:bold">Contact Email</td>
              <td style="padding:6px">{person.get('email','—')}</td></tr>
        </table>
        <p>Please find the full health risk report attached.</p>
      </div>
    </body></html>
    """
    _send_single_email(sender, password, authority_email,
                       authority_subject, authority_html, pdf_path)
    print(f" Alert email sent to authority: {authority_email}")


def _send_single_email(sender, password, recipient, subject, html_body, attachment_path=None):
    msg = MIMEMultipart("alternative")
    msg["From"]    = sender
    msg["To"]      = recipient
    msg["Subject"] = subject
    msg.attach(MIMEText(html_body, "html"))

    if attachment_path and os.path.exists(attachment_path):
        with open(attachment_path, "rb") as f:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition",
                        f'attachment; filename="{os.path.basename(attachment_path)}"')
        msg.attach(part)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(sender, password)
        server.sendmail(sender, recipient, msg.as_string())

In [21]:
# DASHBOARD GENERATOR

def generate_dashboard(all_results: list, save_path: str = "health_dashboard.png"):
    """Generate a 2×3 visual dashboard of all processed individuals."""
    if not all_results:
        print("No data to plot.")
        return

    df = pd.DataFrame(all_results)
    fig = plt.figure(figsize=(18, 13))
    fig.patch.set_facecolor("#f7f9fc")
    fig.suptitle("Haryana Water Quality — Health Risk Dashboard",
                 fontsize=17, fontweight="bold", color="#1a252f", y=0.98)

    risk_order  = ["CRITICAL", "HIGH", "MODERATE", "LOW"]
    risk_colors = {"CRITICAL": "#c0392b", "HIGH": "#e67e22",
                   "MODERATE": "#f1c40f", "LOW":  "#27ae60"}

    # Plot 1: Risk level distribution 
    ax1 = fig.add_subplot(2, 3, 1)
    risk_counts = df["risk_level"].value_counts().reindex(risk_order, fill_value=0)
    bars = ax1.bar(risk_counts.index,
                   risk_counts.values,
                   color=[risk_colors[r] for r in risk_counts.index],
                   edgecolor="white", linewidth=1.5)
    for bar, val in zip(bars, risk_counts.values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha="center", fontweight="bold", fontsize=11)
    ax1.set_title("Risk Level Distribution", fontweight="bold", pad=8)
    ax1.set_ylabel("Number of People")
    ax1.set_facecolor("#f0f4f8")
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    #  Plot 2: Risk pie chart 
    ax2 = fig.add_subplot(2, 3, 2)
    nonzero = risk_counts[risk_counts > 0]
    wedges, texts, autotexts = ax2.pie(
        nonzero.values,
        labels=nonzero.index,
        colors=[risk_colors[r] for r in nonzero.index],
        autopct="%1.0f%%",
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 2}
    )
    for at in autotexts:
        at.set_fontsize(8)
        at.set_color("white")
        at.set_fontweight("bold")
    ax2.set_title("Risk Distribution (%)", fontweight="bold", pad=8)

    #  Plot 3: Risk score per person 
    ax3 = fig.add_subplot(2, 3, 3)
    names  = [str(r["name"])[:12] for r in all_results]
    scores = [r["risk_score"] for r in all_results]
    levels = [r["risk_level"] for r in all_results]
    bar_colors = [risk_colors[l] for l in levels]
    bars3 = ax3.barh(names, scores, color=bar_colors, edgecolor="white")
    for bar, sc in zip(bars3, scores):
        ax3.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 str(sc), va="center", fontsize=9, fontweight="bold")
    ax3.set_title("Individual Risk Scores", fontweight="bold", pad=8)
    ax3.set_xlabel("Risk Score")
    ax3.set_facecolor("#f0f4f8")
    ax3.spines["top"].set_visible(False)
    ax3.spines["right"].set_visible(False)

    # Plot 4: Symptom frequency bar 
    ax4 = fig.add_subplot(2, 3, 4)
    all_symptoms = []
    for r in all_results:
        all_symptoms.extend(r.get("symptoms", []))
    if all_symptoms:
        sym_series = pd.Series(all_symptoms).value_counts().head(10)
        ax4.barh(sym_series.index[::-1], sym_series.values[::-1],
                 color="#e74c3c", alpha=0.8, edgecolor="white")
        ax4.set_title("Top Reported Symptoms", fontweight="bold", pad=8)
        ax4.set_xlabel("Number of People")
        ax4.set_facecolor("#f0f4f8")
        ax4.spines["top"].set_visible(False)
        ax4.spines["right"].set_visible(False)
    else:
        ax4.text(0.5, 0.5, "No symptoms data", ha="center", va="center",
                 transform=ax4.transAxes, color="grey")
        ax4.set_title("Top Reported Symptoms", fontweight="bold", pad=8)

    #  Plot 5: WQI class of nearest sites 
    ax5 = fig.add_subplot(2, 3, 5)
    wqi_class_colors = {
        "Excellent": "#2ecc71", "Good": "#f1c40f",
        "Poor": "#e67e22", "Very Poor": "#e74c3c",
        "Unsuitable": "#8e44ad", "Unknown": "#95a5a6"
    }
    wqi_counts = df["wqi_class"].value_counts()
    ax5.bar(wqi_counts.index,
            wqi_counts.values,
            color=[wqi_class_colors.get(c, "#95a5a6") for c in wqi_counts.index],
            edgecolor="white", linewidth=1.5)
    for i, (idx, val) in enumerate(wqi_counts.items()):
        ax5.text(i, val + 0.05, str(val), ha="center", fontweight="bold")
    ax5.set_title("Nearest Site WQI Class", fontweight="bold", pad=8)
    ax5.set_ylabel("Number of People")
    ax5.tick_params(axis="x", rotation=20)
    ax5.set_facecolor("#f0f4f8")
    ax5.spines["top"].set_visible(False)
    ax5.spines["right"].set_visible(False)

    # Plot 6: Risk flags summary table 
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis("off")
    flagged = [r for r in all_results if r["risk_level"] in ("CRITICAL", "HIGH")]
    table_data = [["Name", "Risk", "Site", "Symptoms"]]
    for r in flagged[:8]:
        table_data.append([
            str(r["name"])[:14],
            r["risk_level"],
            str(r.get("nearest_site","—"))[:14],
            str(len(r.get("symptoms",[])))
        ])
    if not flagged:
        table_data.append(["—", "No high-risk", "individuals", "found"])

    tbl = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                    loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1.1, 1.5)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="white", fontweight="bold")
        elif col == 1 and row > 0:
            level = table_data[row][1]
            cell.set_facecolor(risk_colors.get(level, "#eee"))
            cell.set_text_props(color="white", fontweight="bold")
        else:
            cell.set_facecolor("#f8f9fa" if row % 2 == 0 else "white")
        cell.set_edgecolor("#ddd")
    ax6.set_title("Flagged Individuals (CRITICAL + HIGH)",
                  fontweight="bold", pad=8)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"\n Dashboard saved: {save_path}")

In [22]:
# MAIN PIPELINE

def run_health_alert_pipeline(health_records: list,
                               send_emails: bool = True,
                               output_dir: str = "health_reports"):
    """
    Process a list of health records, generate PDFs, send emails, show dashboard.

    Parameters
    ----------
    health_records : list of dicts — one dict per person (see HEALTH_DATA below)
    send_emails    : set False to skip email (e.g. during testing)
    output_dir     : folder to save PDFs and dashboard
    """
    os.makedirs(output_dir, exist_ok=True)
    all_results = []
    flagged_count = 0

    print(f"\n{'='*60}")
    print(f"  HEALTH RISK PROCESSING — {len(health_records)} records")
    print(f"{'='*60}")

    for person in health_records:
        risk = compute_risk(person)
        name = person.get("name", "Unknown").replace(" ", "_")

        print(f"\n[{risk['risk_level']:8s}] {person.get('name','—'):20s} "
              f"| Score: {risk['risk_score']:3d} "
              f"| Site: {person.get('nearest_site','—')}")
        if risk["triggers"]:
            for t in risk["triggers"]:
                print(f"           ↳ {t}")

        # Generate PDF for HIGH and CRITICAL only
        pdf_path = None
        if risk["risk_level"] in ("CRITICAL", "HIGH"):
            flagged_count += 1
            pdf_path = os.path.join(
                output_dir,
                f"health_report_{name}_{risk['risk_level']}.pdf"
            )
            generate_pdf_report(person, risk, pdf_path)
            if send_emails:
                send_alert_email(person, risk, pdf_path)

        all_results.append({
            "name":         person.get("name", "—"),
            "age":          person.get("age", "—"),
            "risk_level":   risk["risk_level"],
            "risk_score":   risk["risk_score"],
            "wqi_class":    risk["wqi_class"],
            "nearest_site": person.get("nearest_site", "—"),
            "symptoms":     person.get("_reported_symptoms", []),
            "triggers":     risk["triggers"],
        })

    # Dashboard
    dash_path = os.path.join(output_dir, "health_dashboard.png")
    generate_dashboard(all_results, save_path=dash_path)

    # Summary
    print(f"\n{'='*60}")
    print(f"  SUMMARY")
    print(f"  Total records processed : {len(health_records)}")
    print(f"  Flagged (HIGH/CRITICAL) : {flagged_count}")
    print(f"  PDFs generated          : {flagged_count}")
    print(f"  Output folder           : {output_dir}/")
    print(f"{'='*60}\n")

    return pd.DataFrame(all_results)

In [23]:
# =============================================================================
#  ENTER YOUR HEALTH DATA HERE
#   One dict per person — fill in the values from your questionnaire responses
#   All symptom fields: True = symptom present, False = not present
# =============================================================================
HEALTH_DATA = [
    {
        #  Personal Info (Q1–Q6) 
        "name":           "Rahul Sharma",
        "age":            35,
        "gender":         "Male",
        "education":      "Matric",
        "family_type":    "Joint",
        "family_size":    6,
        "employment":     "Employed-Govt",
        "email":          "rahul.sharma@example.com",

        # Location 
        # Use the site name from your data (e.g. "Ratia", "Faridabad", etc.)
        "nearest_site":   "Ratia",

        #  Sanitation & Hygiene (Q10–Q20) 
        "drinking_water_source":   "Tube well",      # river/well/tube well/tap water
        "drinking_water_quality":  "Satisfactory",   # satisfactory/unsatisfactory
        "uses_purifier":           "No",             # Yes/No
        "industrial_waste_mixing": "Yes",            # Yes/No (Q13)
        "has_toilet":              "Yes",            # Yes/No
        "toilet_type":             "Private",        # private/public/open fields
        "uses_disinfectant":       "No",             # Yes/No (Q18)
        "washes_hands":            "Yes",            # Yes/No (Q16)
        "kitchen_fuel":            "LPG",            # cow dung/wood/coal/kerosene/LPG
        "wash_frequency":          "Always",         # Always/Sometimes/Never

        #  Health Status (Q21–Q25) 
        "health_status":           "Bad",            # Good/Bad
        "had_health_issue":        "Yes",            # Yes/No

        #  Symptoms (Q25) — set True if the person reported this symptom 
        "diarrhea":            False,
        "diarrhea_blood":      False,
        "kidney_disease":      True,
        "jaundice":            False,
        "dysentery":           False,
        "fever":               True,
        "anemia":              False,
        "high_blood_pressure": True,
        "vomiting":            False,
        "abdominal_pain":      True,
        "nausea":              False,
        "weight_loss":         False,
        "painful_skin_lesions":False,
        "bone_disease":        False,
        "malnutrition":        False,
        "growth_retardation":  False,
        "gastric_problems":    True,
        "severe_throat":       False,
        "depression":          False,
        "menstrual_problems":  False,

        #  Follow-up (Q26–Q29) 
        "visited_doctor":      "Yes",
        "on_medication":       "Yes",
        "health_recovered":    "No",
        "neighborhood_issues": "Yes",
    },
    {
        "name":           "Priya Devi",
        "age":            28,
        "gender":         "Female",
        "education":      "Graduation",
        "family_type":    "Unit",
        "family_size":    4,
        "employment":     "Unemployed",
        "email":          "priya.devi@example.com",
        "nearest_site":   "Faridabad",

        "drinking_water_source":   "River water",
        "drinking_water_quality":  "Unsatisfactory",
        "uses_purifier":           "No",
        "industrial_waste_mixing": "Yes",
        "has_toilet":              "No",
        "toilet_type":             "Open Fields",
        "uses_disinfectant":       "No",
        "washes_hands":            "No",
        "kitchen_fuel":            "Cow dung cakes",
        "wash_frequency":          "Never",

        "health_status":           "Bad",
        "had_health_issue":        "Yes",

        "diarrhea":            True,
        "diarrhea_blood":      True,
        "kidney_disease":      False,
        "jaundice":            True,
        "dysentery":           True,
        "fever":               True,
        "anemia":              True,
        "high_blood_pressure": False,
        "vomiting":            True,
        "abdominal_pain":      True,
        "nausea":              True,
        "weight_loss":         True,
        "painful_skin_lesions":False,
        "bone_disease":        False,
        "malnutrition":        True,
        "growth_retardation":  False,
        "gastric_problems":    True,
        "severe_throat":       False,
        "depression":          True,
        "menstrual_problems":  True,

        "visited_doctor":      "No",
        "on_medication":       "No",
        "health_recovered":    "No",
        "neighborhood_issues": "Yes",
    },
    {
        "name":           "Sukhwinder Singh",
        "age":            52,
        "gender":         "Male",
        "education":      "PG",
        "family_type":    "Joint",
        "family_size":    8,
        "employment":     "Self-financed business",
        "email":          "sukhwinder@example.com",
        "nearest_site":   "Kaushalaya Dam",

        "drinking_water_source":   "Tap water",
        "drinking_water_quality":  "Satisfactory",
        "uses_purifier":           "Yes",
        "industrial_waste_mixing": "No",
        "has_toilet":              "Yes",
        "toilet_type":             "Private",
        "uses_disinfectant":       "Yes",
        "washes_hands":            "Yes",
        "kitchen_fuel":            "LPG",
        "wash_frequency":          "Always",

        "health_status":           "Good",
        "had_health_issue":        "No",

        "diarrhea":            False,
        "diarrhea_blood":      False,
        "kidney_disease":      False,
        "jaundice":            False,
        "dysentery":           False,
        "fever":               False,
        "anemia":              False,
        "high_blood_pressure": True,
        "vomiting":            False,
        "abdominal_pain":      False,
        "nausea":              False,
        "weight_loss":         False,
        "painful_skin_lesions":False,
        "bone_disease":        False,
        "malnutrition":        False,
        "growth_retardation":  False,
        "gastric_problems":    False,
        "severe_throat":       False,
        "depression":          False,
        "menstrual_problems":  False,

        "visited_doctor":      "Yes",
        "on_medication":       "Yes",
        "health_recovered":    "Yes",
        "neighborhood_issues": "No",
    },
    # ── ADD MORE PEOPLE BELOW — copy a block above and fill in values ────────
    {
        "name":           "Rajveer",
        "age":            30,
        "gender":         "Male",
        "education":      "Post- Graduation",
        "family_type":    "Unit",
        "family_size":    4,
        "employment":     "Employed",
        "email":          "rajveer@example.com",
        "nearest_site":   "Faridabad",

        "drinking_water_source":   "River water",
        "drinking_water_quality":  "Unsatisfactory",
        "uses_purifier":           "Yes",
        "industrial_waste_mixing": "Yes",
        "has_toilet":              "Yes",
        "toilet_type":             "Toilet at home",
        "uses_disinfectant":       "Yes",
        "washes_hands":            "Yes",
        "kitchen_fuel":            "LPG",
        "wash_frequency":          "Never",

        "health_status":           "Good",
        "had_health_issue":        "No",

        "diarrhea":            False,
        "diarrhea_blood":      False,
        "kidney_disease":      False,
        "jaundice":            True,
        "dysentery":           True,
        "fever":               True,
        "anemia":              False,
        "high_blood_pressure": False,
        "vomiting":            False,
        "abdominal_pain":      True,
        "nausea":              True,
        "weight_loss":         False,
        "painful_skin_lesions":False,
        "bone_disease":        False,
        "malnutrition":        False,
        "growth_retardation":  False,
        "gastric_problems":    True,
        "severe_throat":       False,
        "depression":          True,
        "menstrual_problems":  False,

        "visited_doctor":      "Yes",
        "on_medication":       "No",
        "health_recovered":    "No",
        "neighborhood_issues": "Yes",
    },
    # ── ADD MORE PEOPLE BELOW — copy a block above and fill in values ────────
    {
        "name":           "Kiran",
        "age":            19,
        "gender":         "Female",
        "education":      "Under graduate",
        "family_type":    "Joint",
        "family_size":    11,
        "employment":     "Unemployed",
        "email":          "kiran@example.com",
        "nearest_site":   "Faridabad",

        "drinking_water_source":   "River water",
        "drinking_water_quality":  "Unsatisfactory",
        "uses_purifier":           "No",
        "industrial_waste_mixing": "Yes",
        "has_toilet":              "Yes",
        "toilet_type":             "Toilet at home",
        "uses_disinfectant":       "No",
        "washes_hands":            "Yes",
        "kitchen_fuel":            "LPG",
        "wash_frequency":          "Never",

        "health_status":           "Good",
        "had_health_issue":        "No",

        "diarrhea":            False,
        "diarrhea_blood":      False,
        "kidney_disease":      False,
        "jaundice":            False,
        "dysentery":           False,
        "fever":               True,
        "anemia":              False,
        "high_blood_pressure": False,
        "vomiting":            False,
        "abdominal_pain":      False,
        "nausea":              False,
        "weight_loss":         False,
        "painful_skin_lesions":False,
        "bone_disease":        False,
        "malnutrition":        False,
        "growth_retardation":  False,
        "gastric_problems":    True,
        "severe_throat":       False,
        "depression":          False,
        "menstrual_problems":  True,

        "visited_doctor":      "No",
        "on_medication":       "No",
        "health_recovered":    "No",
        "neighborhood_issues": "Yes",
    },
     # ── ADD MORE PEOPLE BELOW — copy a block above and fill in values ────────
    {
        "name":           "Manoj",
        "age":            49,
        "gender":         "Male",
        "education":      "Graduate",
        "family_type":    "Joint",
        "family_size":    14,
        "employment":     "Employed",
        "email":          "manoj@example.com",
        "nearest_site":   "Panchkula",

        "drinking_water_source":   "Surface water",
        "drinking_water_quality":  "Satisfactory",
        "uses_purifier":           "Yes",
        "industrial_waste_mixing": "No",
        "has_toilet":              "Yes",
        "toilet_type":             "Toilet at home",
        "uses_disinfectant":       "Yes",
        "washes_hands":            "Yes",
        "kitchen_fuel":            "LPG",
        "wash_frequency":          "Regular",

        "health_status":           "Good",
        "had_health_issue":        "No",

        "diarrhea":            False,
        "diarrhea_blood":      False,
        "kidney_disease":      False,
        "jaundice":            False,
        "dysentery":           False,
        "fever":               False,
        "anemia":              False,
        "high_blood_pressure": False,
        "vomiting":            False,
        "abdominal_pain":      False,
        "nausea":              False,
        "weight_loss":         False,
        "painful_skin_lesions":False,
        "bone_disease":        False,
        "malnutrition":        False,
        "growth_retardation":  False,
        "gastric_problems":    False,
        "severe_throat":       False,
        "depression":          False,
        "menstrual_problems":  False,

        "visited_doctor":      "Yes",
        "on_medication":       "Yes",
        "health_recovered":    "Yes",
        "neighborhood_issues": "No",
    },
]

In [24]:
# RUN THE PIPELINE

if __name__ == "__main__":
    results_df = run_health_alert_pipeline(
        HEALTH_DATA,
        send_emails=True,    # ← set False to skip emails during testing
        output_dir="health_reports"
    )
    print("\nFull Results Table:")
    print(results_df[["name", "risk_level", "risk_score",
                       "wqi_class", "nearest_site"]].to_string(index=False))


  HEALTH RISK PROCESSING — 6 records

[CRITICAL] Rahul Sharma         | Score:  54 | Site: Ratia
           ↳ Lives near Very Poor WQI site (Ratia)
           ↳ Drinking from unprotected water source (river/open well)
           ↳ No water purifier used despite living near polluted site
           ↳ 5 health symptoms reported (severity score: 14)
 PDF saved: health_reports\health_report_Rahul_Sharma_CRITICAL.pdf
  ⚠️  Email credentials not set. Skipping email.
     Set ALERT_EMAIL_SENDER and ALERT_EMAIL_PASSWORD env variables.

[CRITICAL] Priya Devi           | Score:  99 | Site: Faridabad
           ↳ Lives near Unsuitable WQI site (Faridabad)
           ↳ Drinking from unprotected water source (river/open well)
           ↳ Drinking water quality reported as unsatisfactory/unfiltered
           ↳ No access to private toilet (open defecation)
           ↳ No water purifier used despite living near polluted site
           ↳ 14 health symptoms reported (severity score: 39)
 PDF saved: